In [1]:
import re
from collections import Counter

import nltk
import numpy as np
import pandas as pd
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package punkt to /home/arthurpmrs/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/arthurpmrs/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/arthurpmrs/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/arthurpmrs/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [3]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def preprocess_lyrics(text):
    if pd.isna(text):
        return ""

    text = text.lower()
    text = re.sub(r"\b\w*\d\w*\b", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)
    stems = [
        stemmer.stem(token)
        for token in tokens
        if token not in stop_words
    ]

    return " ".join(stems)

def get_word_count(text: str) -> dict:
    tokens = text.split()
    return Counter(token for token in tokens).most_common()

def get_similarity_result(X_count, selected_indices) -> pd.DataFrame:
    results_count = []

    for idx in selected_indices:
        similarities = cosine_similarity(X_count[idx],X_count).flatten()

        similarities[idx] = -1

        most_similar_position = similarities.argmax()
        similarity_score = similarities[most_similar_position]

        results_count.append({
            "selected_index": idx,
            "similar_index": most_similar_position,
            "similarity": similarity_score
        })

    return pd.DataFrame(results_count).sort_values("similarity", ascending=False).reset_index(drop=True)

In [4]:
# Obter letras pre-processadas
df = pd.read_csv("data/songs_reduced.csv")
df["preprocessed_lyrics"] = df["lyrics"].apply(preprocess_lyrics)
texts = df["preprocessed_lyrics"].fillna("")

In [5]:
# Selecionar 5 músicas aleatoriamente
selected_indices = df.sample(5, random_state=42).index

selected = df.loc[selected_indices]

selected[["artists", "name", "preprocessed_lyrics"]]

,artists,name,preprocessed_lyrics
10650,"[""The Strumbellas""]",In This Life,know season aint chang everyday look like rain...
2041,"[""Therion""]",Land of Canaan,noah curs canaan live servant teshub anger sto...
8668,"[""Damian Marley""]",Julie,juli one truli one love best juli break heart ...
1114,"[""The Beatles""]",Norwegian Wood (This Bird Has Flown) - Remaste...,girl say show room isnt good norwegian wood as...
13902,"[""Demons & Wizards""]",Final Warning,doom testifi there danc death pass pump wave b...


## Letra A - CountVectorizer
Buscar letras mais semelhantes com base em CountVectorizer + Similaridade de Cossenos

In [10]:
# Calcular matriz de documentos
count_vectorizer = CountVectorizer()
X_count = count_vectorizer.fit_transform(texts)
print(X_count.shape)
pd.DataFrame(X_count.toarray(), columns=count_vectorizer.get_feature_names_out())

(20000, 66420)


,___,_____,_______,_________,_aliens_,_con,_def,_differ,_fact,_foundation_,...,힘들어져,힘들어하고,힘들어할,힘을,힙합,힙합은,힙합의,ﬁnd,ﬁndin,ﬁre
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
19996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
19997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
19998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
# Calcular diferença de cossenos entre as 5 músicas selecionadas e todas as demais,
# excluíndo ela própria (Nota dada para a similaridade com ela mesma = -1)
results_count = get_similarity_result(X_count, selected_indices)
results_count

,selected_index,similar_index,similarity
0,1114,17919,1.000000
1,8668,10392,0.524894
2,10650,7011,0.486212
3,2041,8180,0.305823
4,13902,13903,0.275174


In [ ]:
# Detalhes sobre as músicas escolhidas e as selecionadas como mais parecidas
print("-" * 60)
for _, row in results_count.iterrows():
    original = df.loc[row["selected_index"]]
    similar = df.loc[row["similar_index"]]

    print(f"Música escolhida: {original['name']} ({original['artists']})")
    print(f"Mais semelhante:  {similar['name']} ({similar['artists']})")
    print(f"Similaridade:     {row['similarity']:.4f}")
    print("-" * 60)

------------------------------------------------------------
Música escolhida: Norwegian Wood (This Bird Has Flown) - Remastered 2009 (["The Beatles"])
Mais semelhante:  Norwegian Wood (["Waylon Jennings"])
Similaridade:     1.0000
------------------------------------------------------------
Música escolhida: Julie (["Damian Marley"])
Mais semelhante:  One More (["Jimmy Cliff"])
Similaridade:     0.5249
------------------------------------------------------------
Música escolhida: In This Life (["The Strumbellas"])
Mais semelhante:  Something About You (["James Carter", "BCS"])
Similaridade:     0.4862
------------------------------------------------------------
Música escolhida: Land of Canaan (["Therion"])
Mais semelhante:  Awesome God / How Great Is Our God (["Triumphant Quartet"])
Similaridade:     0.3058
------------------------------------------------------------
Música escolhida: Final Warning (["Demons & Wizards"])
Mais semelhante:  I'm Comin' On Back To You (["Jackie Wilson"])

Com base nos dados apresentados acima, é possível perceber que uma das músicas escolhidas, "Norwegian Wood (This Bird Has Flown) - Remastered 2009" encontrou um par de semelhança perfeita, Norwegian Wood, apresentando coeficiente de similaridade igual a 1.

Apesar de a similaridade com o próprio texto não ter sido considerada como pede a atividade, aconteceu de selecionarmos (aleatoriamente) uma música que possuía uma regravação dentro da base. Como se trata de uma regravação, a musicalidade e o artista são diferentes, mas a letra é exatamente igual. Se a letra é exatamente igual, os vetores obtidos pelo CountVectorize são também exatamente iguais, ou seja, possuem um ângulo entre eles igual a 0º e o cosseno de 0º é 1.

O segundo par de maior similaridade fora mas músicas "Julie" e "One More". Para verificar porque isso acontece, vamos listas os tokens mais comuns entre os pares de música, desconsiderando o caso de similaridade igual a 1.

In [16]:
def compare_cv_count_tokens(selected_index, similar_index, X_count, count_vectorizer):
    feature_names = count_vectorizer.get_feature_names_out()

    selected_counts = X_count[selected_index].toarray().ravel()
    similar_counts = X_count[similar_index].toarray().ravel()

    comparison = pd.DataFrame({
        "token": feature_names,
        "selected_count": selected_counts,
        "similar_count": similar_counts
    })

    comparison = comparison[
        (comparison["selected_count"] > 0) & (comparison["similar_count"] > 0)
    ].copy()
    comparison["count_product"] = comparison["selected_count"] * comparison["similar_count"]

    return comparison.sort_values("count_product", ascending=False).reset_index(drop=True)

In [20]:
# Apresentar os resultados apenas para os casos em que as letras não eram iguais
filtered_results_count = results_count[results_count['similarity'] < 0.999]
for _, row in filtered_results_count.iterrows():
    selected_index = int(row["selected_index"])
    similar_index = int(row["similar_index"])

    original_name = df.iloc[selected_index]["name"]
    similar_name = df.iloc[similar_index]["name"]

    comparison = compare_cv_count_tokens(
        selected_index,
        similar_index,
        X_count,
        count_vectorizer
    )

    print("-" * 60)
    print(f"{original_name} × {similar_name}")
    print(f"Similaridade: {row['similarity']:.4f}")
    print(f"Quantidade de tokens compartilhados: {len(comparison)}")

    display(comparison.head(10))

------------------------------------------------------------
Julie × One More
Similaridade: 0.5249
Quantidade de tokens compartilhados: 8


,token,selected_count,similar_count,count_product
0,one,17,47,799
1,im,4,2,8
2,run,4,2,8
3,away,4,1,4
4,love,4,1,4
5,babi,4,1,4
6,say,4,1,4
7,give,1,1,1


------------------------------------------------------------
In This Life × Something About You
Similaridade: 0.4862
Quantidade de tokens compartilhados: 3


,token,selected_count,similar_count,count_product
0,someth,10,35,350
1,there,10,9,90
2,keep,1,1,1


------------------------------------------------------------
Land of Canaan × Awesome God / How Great Is Our God
Similaridade: 0.3058
Quantidade de tokens compartilhados: 6


,token,selected_count,similar_count,count_product
0,god,8,32,256
1,great,2,10,20
2,see,4,2,8
3,heaven,1,4,4
4,sky,2,1,2
5,night,1,1,1


------------------------------------------------------------
Final Warning × I'm Comin' On Back To You
Similaridade: 0.2752
Quantidade de tokens compartilhados: 11


,token,selected_count,similar_count,count_product
0,come,4,26,104
1,im,3,14,42
2,back,1,4,4
3,oh,1,3,3
4,let,3,1,3
5,hold,1,2,2
6,ye,2,1,2
7,take,1,1,1
8,there,1,1,1
9,think,1,1,1


A similaridade de cossenos calcula o produto interno entre os vetores e divide pelo produto das normas.

$\cos(\theta) = \frac{A \cdot B}{||A|| ||B||}$

Ou seja, dado os vetores $a = (a_1, a_2)$ e $b = (b_1, b_2)$, será calculado $a_1 b_1 + a_2 b_2$. No caso do `CountVectorizer`, $a_i$ e $b_i$ são as quantidades de um determinado token em cada texto. Portanto, quando um token $i$ se repete muitas vezes nos dois textos, o produto entre suas contagens aumenta ($a_i b_i$), aumentando também a similaridade.

No caso do par "Julie" e "One More", o token "one" apareceu 17 vezes em uma das músicas e 47 na outra. Isso resultou no maior produto de contagens dentre as músicas selecionadas (799), o que ajuda a explicar porque a similaridade de cosseno aplicada aos vetores do `CountVectorizer` identificou esse par como o mais semelhante.

Ou seja, aplicar a similaridade de cosseno com os vetores do `CountVectorizer` resulta em uma análise da frequência dos tokens em cada texto, dando maior similaridade aos textos que possuem um padrão de frequência mais próximo, já que também é feita a divisão do produto interno pela norma.

## Letra B - TF-IDF
Buscar letras mais semelhantes com base em TF-IDF + Similaridade de Cossenos

In [ ]:
# Calcular matriz de documentos
tfidf_vectorizer = TfidfVectorizer(
    norm="l2",
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=False
)

X_tfidf = tfidf_vectorizer.fit_transform(texts)

print(X_tfidf.shape)

pd.DataFrame(X_tfidf.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

(20000, 66420)


,___,_____,_______,_________,_aliens_,_con,_def,_differ,_fact,_foundation_,...,힘들어져,힘들어하고,힘들어할,힘을,힙합,힙합은,힙합의,ﬁnd,ﬁndin,ﬁre
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19996,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19997,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
# Calcular diferença de cossenos entre as 5 músicas selecionadas e todas as demais
results_tfidf = get_similarity_result(X_tfidf, selected_indices)
results_tfidf

,selected_index,similar_index,similarity
0,1114,17919,1.000000
1,10650,7011,0.501807
2,8668,4475,0.389407
3,2041,14338,0.275371
4,13902,18924,0.206906


In [8]:
print("-" * 60)

for _, row in results_tfidf.iterrows():
    selected_index = int(row["selected_index"])
    similar_index = int(row["similar_index"])

    original = df.iloc[selected_index]
    similar = df.iloc[similar_index]

    print(
        f"Música escolhida: {original['name']} "
        f"({original['artists']})"
    )
    print(
        f"Mais semelhante:  {similar['name']} "
        f"({similar['artists']})"
    )
    print(f"Similaridade:      {row['similarity']:.4f}")
    print("-" * 60)

------------------------------------------------------------
Música escolhida: Norwegian Wood (This Bird Has Flown) - Remastered 2009 (["The Beatles"])
Mais semelhante:  Norwegian Wood (["Waylon Jennings"])
Similaridade:      1.0000
------------------------------------------------------------
Música escolhida: In This Life (["The Strumbellas"])
Mais semelhante:  Something About You (["James Carter", "BCS"])
Similaridade:      0.5018
------------------------------------------------------------
Música escolhida: Julie (["Damian Marley"])
Mais semelhante:  1000 Julys (["Third Eye Blind"])
Similaridade:      0.3894
------------------------------------------------------------
Música escolhida: Land of Canaan (["Therion"])
Mais semelhante:  Art of Scratch (Intro) (["Artifacts"])
Similaridade:      0.2754
------------------------------------------------------------
Música escolhida: Final Warning (["Demons & Wizards"])
Mais semelhante:  Praise Like Fireworks (["Rend Collective"])
Similaridade

In [13]:
def compare_tfidf_tokens(selected_index, similar_index, X_tfidf, tfidf_vectorizer):
    feature_names = tfidf_vectorizer.get_feature_names_out()

    selected_weights = X_tfidf[
        selected_index
    ].toarray().ravel()

    similar_weights = X_tfidf[
        similar_index
    ].toarray().ravel()

    comparison = pd.DataFrame({
        "token": feature_names,
        "idf": tfidf_vectorizer.idf_,
        "selected_tfidf": selected_weights,
        "similar_tfidf": similar_weights
    })

    comparison = comparison[
        (comparison["selected_tfidf"] > 0) & (comparison["similar_tfidf"] > 0)
    ].copy()

    comparison["weight_product"] = (
        comparison["selected_tfidf"] * comparison["similar_tfidf"]
    )

    selected_norm = np.linalg.norm(selected_weights)
    similar_norm = np.linalg.norm(similar_weights)

    comparison["cosine_contribution"] = (
        comparison["weight_product"] /
        (selected_norm * similar_norm)
    )

    return comparison.sort_values("cosine_contribution", ascending=False).reset_index(drop=True)

In [15]:
filtered_results_tfidf = results_tfidf[results_tfidf["similarity"] < 0.999]

for _, row in filtered_results_tfidf.iterrows():
    selected_index = int(row["selected_index"])
    similar_index = int(row["similar_index"])

    original_name = df.iloc[selected_index]["name"]
    similar_name = df.iloc[similar_index]["name"]

    comparison = compare_tfidf_tokens(
        selected_index,
        similar_index,
        X_tfidf,
        tfidf_vectorizer
    )

    print("=" * 60)
    print(f"{original_name} × {similar_name}")
    print(f"Similaridade: {row['similarity']:.4f}")
    print(f"Tokens compartilhados: {len(comparison)}")

    display(comparison.head(15))

In This Life × Something About You
Similaridade: 0.5018
Tokens compartilhados: 3


,token,idf,selected_tfidf,similar_tfidf,weight_product,cosine_contribution
0,someth,3.262934,0.481404,0.880764,0.424003,0.424003
1,there,2.743305,0.404739,0.190415,0.077068,0.077068
2,keep,2.543466,0.037526,0.019616,0.000736,0.000736


Julie × 1000 Julys
Similaridade: 0.3894
Tokens compartilhados: 9


,token,idf,selected_tfidf,similar_tfidf,weight_product,cosine_contribution
0,juli,6.684030,0.691037,0.500996,0.346206,0.346206
1,come,1.985959,0.102660,0.124047,0.012735,0.012735
2,babi,2.664736,0.078713,0.099867,0.007861,0.007861
3,mind,2.758858,0.101867,0.068929,0.007022,0.007022
4,im,1.621342,0.047893,0.121526,0.005820,0.005820
5,say,2.118457,0.062577,0.079394,0.004968,0.004968
6,back,2.171878,0.032077,0.081396,0.002611,0.002611
7,even,2.898504,0.042809,0.036209,0.001550,0.001550
8,give,2.620802,0.019354,0.032740,0.000634,0.000634


Land of Canaan × Art of Scratch (Intro)
Similaridade: 0.2754
Tokens compartilhados: 3


,token,idf,selected_tfidf,similar_tfidf,weight_product,cosine_contribution
0,el,6.298367,0.369754,0.730441,0.270084,0.270084
1,peac,4.154778,0.027101,0.172087,0.004664,0.004664
2,one,1.961338,0.012794,0.048742,0.000624,0.000624


Final Warning × Praise Like Fireworks
Similaridade: 0.2069
Tokens compartilhados: 6


,token,idf,selected_tfidf,similar_tfidf,weight_product,cosine_contribution
0,given,5.272326,0.323777,0.606742,0.196449,0.196449
1,heart,2.507608,0.025666,0.240481,0.006172,0.006172
2,let,2.197378,0.067471,0.028097,0.001896,0.001896
3,wont,2.841420,0.029082,0.036332,0.001057,0.001057
4,ill,2.421972,0.024789,0.030969,0.000768,0.000768
5,never,2.076803,0.021256,0.026556,0.000564,0.000564


## Justificativa da IA - ALTERAR
Diferentemente do `CountVectorizer`, que representa os documentos utilizando somente a frequência absoluta dos tokens, o TF-IDF combina a frequência de cada token no documento com sua raridade na coleção. Dessa forma, tokens que aparecem em muitos documentos recebem um peso menor, enquanto tokens mais específicos recebem maior importância.

A similaridade do cosseno aplicada aos vetores TF-IDF continua sendo calculada pelo produto interno dividido pelo produto das normas. Entretanto, cada produto \(a_i b_i\) passa a representar o produto dos pesos TF-IDF de um token compartilhado, e não mais o produto de suas contagens brutas.

Assim, os tokens que mais justificam a similaridade entre um par são aqueles que aparecem nos dois documentos e possuem pesos TF-IDF elevados em ambos. Isso normalmente ocorre quando o token é relevante para esses textos, mas relativamente raro no restante da coleção. Em contrapartida, palavras muito comuns podem ter contribuído bastante na representação do `CountVectorizer`, por causa de suas repetições, mas recebem menor importância no TF-IDF.

Portanto, enquanto o `CountVectorizer` tende a aproximar documentos com padrões semelhantes de frequência, o TF-IDF tende a valorizar a presença compartilhada de termos mais característicos. Essa diferença pode fazer com que os dois métodos selecionem documentos distintos como os mais semelhantes.